In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'Nu', 'Nu_credit_statement.pdf')

### Testing a set of bank statements

In [ ]:
import pandas as pd
from document_reader import PDFReader
from bank_detector import DefaultBankDetector
from table_boundary_detector import TransactionTableBoundaryDetector
from row_segmenter import TransactionRowSegmenter
from table_reconstructor import TransactionTableReconstructor
from table_normalizer import TransactionTableNormalizer

import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

trials_folder = os.path.join(INPUTS_FOLDER, 'test_files', 'trials')

def extract_transactions(pdf_file: str):
    reader = PDFReader(pdf_file)
    bank_detector = DefaultBankDetector(reader)

    extracted_words = bank_detector.extracted_words
    bank = bank_detector.detect_bank()
    statement_type = bank_detector.detect_statement_type()
    statement_properties = bank_detector.get_statement_properties()

    boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

    start_idx = boundary_detector.start_idx
    end_idx = boundary_detector.end_idx

    if start_idx is None or end_idx is None:
        statement_properties = bank_detector.get_statement_properties(new_credit_format= True)
        boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)
        start_idx = boundary_detector.start_idx
        end_idx = boundary_detector.end_idx

    df_filtered = boundary_detector.get_filtered_table_words()

    row_segmenter = TransactionRowSegmenter(df_filtered, statement_properties)

    column_delimitation = row_segmenter.delimit_column_positions()
    grouped_rows = row_segmenter.group_rows()

    table_reconstructor = TransactionTableReconstructor(grouped_rows, column_delimitation, statement_properties)
    reconstructed_table = table_reconstructor.reconstruct_table()

    normalizer = TransactionTableNormalizer(reconstructed_table, extracted_words, statement_properties)
    normalized_table = normalizer.normalize_table()

    return normalized_table, bank, statement_type

if __name__ == "__main__":
    trials_files = [
        "bbvaCred_ago.pdf",
        "bbvaCred_nov.pdf",
        "bbvaCred_oct.pdf",
        "bbvaCred_sep.pdf",
        "bbvadeb_ago.pdf",
        "bbvadeb_nov.pdf",
        "bbvadeb_oct.pdf",
        "bbvadeb_sep.pdf",
        "citicost_ago.pdf",
        "citicost_nov.pdf",
        "citicost_oct.pdf",
        "citicost_sep.pdf",
        "nucred_ago.pdf",
        "nucred_nov.pdf",
        "nucred_oct.pdf",
        "nucred_sep.pdf",
        "nudeb_ago.pdf",
        "nudeb_oct.pdf",
        "nudeb_sep.pdf"
    ]

    df = pd.DataFrame(columns=['Date', 'Description', 'Amount', 'Type', 'bank', 'statement_type', 'file_name'])

    for file in trials_files:
        try:
            file_path = os.path.join(trials_folder, file)

            normalized_table, bank, statement_type = extract_transactions(file_path)
            normalized_table['bank'] = bank
            normalized_table['statement_type'] = statement_type
            normalized_table['file_name'] = file

            df = pd.concat([df, normalized_table], ignore_index=True)
        except Exception as e:
            print(f"Error processing file {file}: {e}")

    df.to_csv(os.path.join(trials_folder, 'trials_results.csv'), index=False)
    print("Results saved to trials_results.csv")

### Pipeline description

In [2]:
from document_reader import PDFReader
from bank_detector import DefaultBankDetector
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

bank_detector = DefaultBankDetector(PDFReader(statement_path))

extracted_words = bank_detector.extracted_words
extracted_words

,page,text,x0,top,x1,bottom
0,1,"¡Hola,",116.450,176.97,184.188,198.97
1,1,Luis,191.228,176.97,238.638,198.97
2,1,Fernando!,245.678,176.97,365.688,198.97
3,1,Este,116.450,209.97,167.446,231.97
4,1,es,174.486,209.97,200.314,231.97
...,...,...,...,...,...,...
827,5,Administración,255.160,644.00,334.970,654.00
828,5,Tributaria,338.170,644.00,389.380,654.00
829,5,5,518.890,815.60,524.074,823.60
830,5,de,526.634,815.60,537.258,823.60


In [3]:
bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.get_statement_properties()

print(f'{bank} - {statement_type}')

nu - credit


In [4]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

if statement_type == 'credit' and (start_idx is None or end_idx is None):
    print('New format')
    statement_properties = bank_detector.get_statement_properties(new_credit_format= True)
    boundary_detector = TransactionTableBoundaryDetector(extracted_words, statement_properties)
    
    start_idx = boundary_detector.start_idx
    end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table

151 - 394


,page,text,x0,top,x1,bottom
0,2,MONTOS,377.79,293.0,426.54,303.0
1,2,EN,429.74,293.0,445.09,303.0
2,2,PESOS,448.29,293.0,484.77,303.0
3,2,MEXICANOS,487.97,293.0,555.00,303.0
4,2,Saldo,94.54,334.0,124.14,344.0
...,...,...,...,...,...,...
238,3,1.00,347.48,620.0,369.21,630.0
239,3,=,372.41,620.0,379.02,630.0
240,3,$20.09),382.22,620.0,424.53,630.0
241,3,USD,497.81,622.0,520.73,632.0


In [5]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table, statement_properties)

column_delimitation = row_segmenter.delimit_column_positions()
grouped_rows = row_segmenter.group_rows()

print(row_segmenter.row_threshold)
print(column_delimitation)
grouped_rows

16.413793103448278
{'columns': ['TRANSACCIONES', '(DE) (\\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (A) (\\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (\\(\\d{2}) (DÍAS\\))', 'MONTOS EN PESOS MEXICANOS'], 'x0': [40.0, None, 377.79], 'x1': [136.12, None, 555.0]}


,row_group,text,words,top,bottom,page
0,0,MONTOS EN PESOS MEXICANOS,"[(MONTOS, 377.79, 426.54), (EN, 429.74, 445.09...",293.0,303.0,2
1,1,"Saldo inicial del periodo (OCT 2024) $8,970.49","[(Saldo, 94.54, 124.14000000000001), (inicial,...",334.0,344.0,2
2,2,"Pagos a tu tarjeta en el periodo - -$8,980.00","[(Pagos, 94.54, 126.81), (a, 130.0100000000000...",359.0,369.0,2
3,3,"Compras $12,427.60","[(Compras, 94.54, 141.85000000000002), ($12,42...",384.0,394.0,2
4,4,Abonos y devoluciones $0.00,"[(Abonos, 94.54, 135.3), (y, 138.5, 144.75), (...",409.0,419.0,2
5,5,Retiros de efectivo $0.00,"[(Retiros, 94.54, 132.02), (de, 135.2200000000...",434.0,444.0,2
6,6,Comisiones $0.00,"[(Comisiones, 94.54, 155.07000000000002), ($0....",459.0,469.0,2
7,7,Intereses de saldo revolvente $0.00,"[(Intereses, 94.54, 142.19000000000003), (de, ...",484.0,494.0,2
8,8,Intereses de saldo a meses $0.00,"[(Intereses, 94.54, 142.19000000000003), (de, ...",509.0,519.0,2
9,9,IVA $0.00,"[(IVA, 94.54, 113.91000000000001), ($0.00, 524...",534.0,544.0,2


In [6]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, column_delimitation, statement_properties)

table_reconstructor.get_structured_table()

,TRANSACCIONES,(DE) (\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (A) (\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (\(\d{2}) (DÍAS\)),MONTOS EN PESOS MEXICANOS
0,,MONTOS EN PESOS MEXICANOS,
1,Saldo inicial,del periodo (OCT 2024),"$8,970.49"
2,Pagos a tu,tarjeta en el periodo -,"-$8,980.00"
3,Compras,,"$12,427.60"
4,Abonos y,devoluciones,$0.00
5,Retiros de,efectivo,$0.00
6,Comisiones,,$0.00
7,Intereses de,saldo revolvente,$0.00
8,Intereses de,saldo a meses,$0.00
9,IVA,,$0.00


In [7]:
df_structured = table_reconstructor.reconstruct_table()
df_structured

,TRANSACCIONES,(DE) (\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (A) (\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (\(\d{2}) (DÍAS\)),MONTOS EN PESOS MEXICANOS
0,08 OCT,Web Transfer Vision Cr Otros,$400.00
1,09 OCT,Paypal*Google Youtube Otros,$279.00
2,10 OCT,Pago a tu tarjeta de crédito - ¡Muchas gracias!,"-$8,980.00"
3,14 OCT,Paypal *Nintendo Servicio,$699.00
4,15 OCT,Paypal *Itunesappst Ap Otros,$99.00
5,15 OCT,Axa Seg Mit Mn Servicio,"$1,964.78"
6,18 OCT,Paypal*Playstation Otros,"$2,436.42"
7,20 OCT,Amazon Mx Marketplace Servicio,$478.48
8,23 OCT,Telmex Cargo Recurr Servicio,$389.00
9,29 OCT,Tst*Lobsterme 2- Restaurante,$604.88


In [8]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_structured, extracted_words, statement_properties)

df_normalized = normalizer.normalize_table()
df_normalized

,Date,Description,Amount,Type
0,2024-10-08,Web Transfer Vision Cr Otros,-400.00,Cargo
1,2024-10-09,Paypal*Google Youtube Otros,-279.00,Cargo
2,2024-10-10,Pago a tu tarjeta de crédito - ¡Muchas gracias!,8980.00,Abono
3,2024-10-14,Paypal *Nintendo Servicio,-699.00,Cargo
4,2024-10-15,Paypal *Itunesappst Ap Otros,-99.00,Cargo
5,2024-10-15,Axa Seg Mit Mn Servicio,-1964.78,Cargo
6,2024-10-18,Paypal*Playstation Otros,-2436.42,Cargo
7,2024-10-20,Amazon Mx Marketplace Servicio,-478.48,Cargo
8,2024-10-23,Telmex Cargo Recurr Servicio,-389.00,Cargo
9,2024-10-29,Tst*Lobsterme 2- Restaurante,-604.88,Cargo


In [9]:
normalizer.period_idx

12

In [10]:
from data_exporter import CsvExporter
from config import OUTPUTS_FOLDER

exporter = CsvExporter(df_normalized)
file_path = os.path.join(OUTPUTS_FOLDER, bank, f'{bank}_{statement_type}.csv')

exporter.export_to_csv(file_path)

Data exported to c:\Users\mario\Proyectos\Aether\Aether_v1\documents\outputs\nu\nu_credit.csv successfully.
